# 16 Exportar baseline clasico final

Este notebook cierra el baseline clasico de registration.

No calcula nuevos registros. Solo consolida el mejor resultado aceptado por sujeto, segun `15_comparacion_final_registration.ipynb`, en una carpeta unica:

`outputs_baseline_clasico_final`

La idea es que esta carpeta sea el punto de comparacion contra futuros metodos deep learning como VoxelMorph o TransMorph.


In [ ]:
from pathlib import Path
import csv
import json
import shutil

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


BASE_DIR = Path(r'Registration\DeepLearning')
REG_DIR = BASE_DIR / 'Tecnicas_registration'
FINAL_COMPARISON_DIR = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_comparacion_final_registration'
RIGID_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_rigido_mascaras'
AFFINE_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_afin_mascaras'
NONRIGID_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_registro_no_rigido_mascaras'
OUTPUT_ROOT = REG_DIR / '00_baseline_clasico' / 'outputs' / 'outputs_baseline_clasico_final'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Comparacion final:', FINAL_COMPARISON_DIR)
print('Salida baseline:', OUTPUT_ROOT)


## Leer decision final

In [ ]:
def read_csv_dicts(path):
    with Path(path).open('r', newline='', encoding='utf-8') as f:
        return list(csv.DictReader(f))


def write_csv(path, rows, fieldnames):
    with Path(path).open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, '') for key in fieldnames})


def json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {k: json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    return value


final_rows = read_csv_dicts(FINAL_COMPARISON_DIR / 'registration_final_decision_summary.csv')
final_by_subject = {row['subject_id']: row for row in final_rows}
final_rows


## Resolver archivos fuente

Cada metodo guarda sus outputs con nombres distintos. Esta funcion traduce la decision final a rutas concretas.


In [ ]:
def source_files_for_final(subject_id, final_method):
    if final_method == 'rigid':
        subject_dir = RIGID_ROOT / subject_id
        suffix = 'rigid'
        return {
            'registered_he_rgb': subject_dir / f'{subject_id}_he_{suffix}_to_hsi.png',
            'registered_he_mask': subject_dir / f'{subject_id}_he_mask_{suffix}_to_hsi.png',
            'overlay_rgb': subject_dir / f'{subject_id}_overlay_rgb_{suffix}.png',
            'overlay_contours': subject_dir / f'{subject_id}_overlay_contours_{suffix}.png',
            'metrics': subject_dir / f'{subject_id}_{suffix}_metrics.json',
        }

    if final_method == 'affine_final':
        subject_dir = AFFINE_ROOT / subject_id
        suffix = 'affine'
        return {
            'registered_he_rgb': subject_dir / f'{subject_id}_he_{suffix}_to_hsi.png',
            'registered_he_mask': subject_dir / f'{subject_id}_he_mask_{suffix}_to_hsi.png',
            'overlay_rgb': subject_dir / f'{subject_id}_overlay_rgb_{suffix}.png',
            'overlay_contours': subject_dir / f'{subject_id}_overlay_contours_{suffix}.png',
            'metrics': subject_dir / f'{subject_id}_{suffix}_metrics.json',
        }

    if final_method == 'nonrigid_final':
        subject_dir = NONRIGID_ROOT / subject_id
        suffix = 'nonrigid'
        return {
            'registered_he_rgb': subject_dir / f'{subject_id}_he_{suffix}_to_hsi.png',
            'registered_he_mask': subject_dir / f'{subject_id}_he_mask_{suffix}_to_hsi.png',
            'overlay_rgb': subject_dir / f'{subject_id}_overlay_rgb_{suffix}.png',
            'overlay_contours': subject_dir / f'{subject_id}_overlay_contours_{suffix}.png',
            'metrics': subject_dir / f'{subject_id}_{suffix}_metrics.json',
            'flow_magnitude': subject_dir / f'{subject_id}_flow_magnitude.png',
            'flow_npz': subject_dir / f'{subject_id}_dense_flow.npz',
        }

    raise ValueError(f'Metodo final no reconocido para {subject_id}: {final_method}')


def copy_if_exists(src, dst):
    src = Path(src)
    dst = Path(dst)
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return dst


## Exportar baseline final

In [ ]:
manifest = {
    'name': 'baseline_clasico_registration_he_to_hsi',
    'description': 'Mejor resultado clasico aceptado por sujeto para usar como baseline frente a metodos deep learning.',
    'registration_direction': 'he_to_hsi',
    'fixed_image': 'hsi',
    'moving_image': 'he',
    'source_notebooks': [
        str(REG_DIR / '00_baseline_clasico' / 'notebooks' / '12_registro_rigido_mascaras_todos.ipynb'),
        str(REG_DIR / '00_baseline_clasico' / 'notebooks' / '13_registro_afin_mascaras_todos.ipynb'),
        str(REG_DIR / '00_baseline_clasico' / 'notebooks' / '14_registro_no_rigido_mascaras_todos.ipynb'),
        str(REG_DIR / '00_baseline_clasico' / 'notebooks' / '15_comparacion_final_registration.ipynb'),
    ],
    'subjects': {},
}

export_rows = []

for subject_id in SUBJECTS:
    row = final_by_subject[subject_id]
    final_method = row['final_method']
    source_files = source_files_for_final(subject_id, final_method)
    subject_out = OUTPUT_ROOT / subject_id
    subject_out.mkdir(parents=True, exist_ok=True)

    exported_files = {
        'registered_he_rgb': subject_out / f'{subject_id}_he_registered_to_hsi_baseline.png',
        'registered_he_mask': subject_out / f'{subject_id}_he_registered_mask_to_hsi_baseline.png',
        'overlay_rgb': subject_out / f'{subject_id}_baseline_overlay_rgb.png',
        'overlay_contours': subject_out / f'{subject_id}_baseline_overlay_contours.png',
        'metrics': subject_out / f'{subject_id}_baseline_metrics.json',
    }

    for key in ['registered_he_rgb', 'registered_he_mask', 'overlay_rgb', 'overlay_contours', 'metrics']:
        copy_if_exists(source_files[key], exported_files[key])

    if 'flow_magnitude' in source_files:
        exported_files['flow_magnitude'] = subject_out / f'{subject_id}_baseline_flow_magnitude.png'
        exported_files['flow_npz'] = subject_out / f'{subject_id}_baseline_dense_flow.npz'
        copy_if_exists(source_files['flow_magnitude'], exported_files['flow_magnitude'])
        copy_if_exists(source_files['flow_npz'], exported_files['flow_npz'])

    subject_summary = {
        'subject_id': subject_id,
        'final_method': final_method,
        'final_note': row['final_note'],
        'visual_review_flag': row['visual_review_flag'],
        'dice_final': row['dice_final'],
        'iou_final': row['iou_final'],
        'contour_mean_final_px': row['contour_mean_final_px'],
        'contour_p95_final_px': row['contour_p95_final_px'],
        'dice_rigid': row['dice_rigid'],
        'dice_affine_final': row['dice_affine_final'],
        'dice_nonrigid_final': row['dice_nonrigid_final'],
        'nonrigid_flow_p95_px': row['nonrigid_flow_p95_px'],
        'nonrigid_flow_max_px': row['nonrigid_flow_max_px'],
        'target_px_per_cm': row['target_px_per_cm'],
        'files': {key: str(path) for key, path in exported_files.items()},
        'source_files': {key: str(path) for key, path in source_files.items()},
    }

    (subject_out / f'{subject_id}_baseline_summary.json').write_text(
        json.dumps(json_ready(subject_summary), indent=2, ensure_ascii=False),
        encoding='utf-8',
    )

    manifest['subjects'][subject_id] = subject_summary
    export_rows.append({
        'subject_id': subject_id,
        'final_method': final_method,
        'visual_review_flag': row['visual_review_flag'],
        'dice_final': row['dice_final'],
        'iou_final': row['iou_final'],
        'contour_mean_final_px': row['contour_mean_final_px'],
        'contour_p95_final_px': row['contour_p95_final_px'],
        'registered_he_rgb': str(exported_files['registered_he_rgb']),
        'registered_he_mask': str(exported_files['registered_he_mask']),
        'overlay_rgb': str(exported_files['overlay_rgb']),
        'overlay_contours': str(exported_files['overlay_contours']),
    })

manifest_path = OUTPUT_ROOT / 'baseline_clasico_manifest.json'
summary_csv = OUTPUT_ROOT / 'baseline_clasico_summary.csv'
summary_json = OUTPUT_ROOT / 'baseline_clasico_summary.json'

manifest_path.write_text(json.dumps(json_ready(manifest), indent=2, ensure_ascii=False), encoding='utf-8')
summary_json.write_text(json.dumps(json_ready(export_rows), indent=2, ensure_ascii=False), encoding='utf-8')
write_csv(
    summary_csv,
    export_rows,
    [
        'subject_id',
        'final_method',
        'visual_review_flag',
        'dice_final',
        'iou_final',
        'contour_mean_final_px',
        'contour_p95_final_px',
        'registered_he_rgb',
        'registered_he_mask',
        'overlay_rgb',
        'overlay_contours',
    ],
)

print('Manifest:', manifest_path)
print('Resumen CSV:', summary_csv)
print('Resumen JSON:', summary_json)


## Crear montajes finales

In [ ]:
def load_image(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def resize_for_montage(img, target_width=360):
    h, w = img.shape[:2]
    scale = target_width / max(w, 1)
    target_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (target_width, target_h), interpolation=cv2.INTER_AREA)


def add_label(img, label):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 34), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 23), cv2.FONT_HERSHEY_SIMPLEX, 0.52, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_grid(items, cols=3, target_width=360):
    images = []
    for path, label in items:
        images.append(add_label(resize_for_montage(load_image(path), target_width), label))
    rows = int(np.ceil(len(images) / cols))
    max_h = max(img.shape[0] for img in images)
    canvas = np.zeros((rows * max_h, cols * target_width, 3), dtype=np.uint8)
    for idx, img in enumerate(images):
        r = idx // cols
        c = idx % cols
        y = r * max_h
        x = c * target_width
        canvas[y:y + img.shape[0], x:x + img.shape[1]] = img
    return canvas


overlay_items = []
contour_items = []
for row in export_rows:
    label = f"{row['subject_id']} | {row['final_method'].replace('_final', '')} | D {float(row['dice_final']):.3f}"
    overlay_items.append((row['overlay_rgb'], label))
    contour_items.append((row['overlay_contours'], label))

overlay_grid = make_grid(overlay_items)
contour_grid = make_grid(contour_items)

overlay_grid_path = OUTPUT_ROOT / 'baseline_clasico_overlay_grid.png'
contour_grid_path = OUTPUT_ROOT / 'baseline_clasico_contour_grid.png'
Image.fromarray(overlay_grid).save(overlay_grid_path)
Image.fromarray(contour_grid).save(contour_grid_path)

print('Montaje overlay:', overlay_grid_path)
print('Montaje contornos:', contour_grid_path)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
axes[0].imshow(overlay_grid)
axes[0].axis('off')
axes[0].set_title('Baseline clasico final - overlays')
axes[1].imshow(contour_grid)
axes[1].axis('off')
axes[1].set_title('Baseline clasico final - contornos')
plt.show()


## README del baseline

In [ ]:
readme = f"""# Baseline clasico final de registration H&E -> HSI

Esta carpeta contiene el mejor resultado clasico aceptado por sujeto.

Direccion del registro:

- Imagen fija: HSI
- Imagen movil: H&E

Metodos considerados:

1. Rigido por mascaras: rotacion + traslacion.
2. Afin por mascaras: OpenCV ECC sobre mapas de distancia firmados.
3. No rigido: TV-L1 optical flow sobre mapas de distancia firmados.

Regla de decision:

1. Si el no rigido fue aceptado, se usa no rigido.
2. Si no, pero el afin fue aceptado, se usa afin.
3. Si no, se usa rigido.

Archivos principales:

- `baseline_clasico_summary.csv`: resumen por sujeto.
- `baseline_clasico_manifest.json`: manifest completo.
- `baseline_clasico_overlay_grid.png`: montaje de overlays finales.
- `baseline_clasico_contour_grid.png`: montaje de contornos finales.

Flags de revision:

- `ok`: resultado aceptado sin avisos fuertes.
- `review_moderate_nonrigid_displacement`: revisar visualmente por desplazamiento no rigido moderado.
- `review_high_nonrigid_displacement`: revisar con cuidado por desplazamiento no rigido alto.
- `review_rejected_nonrigid`: el no rigido se rechazo; el baseline vuelve a rigido/afin.

Nota importante:

Este baseline registra principalmente la geometria del especimen mediante mascaras/contornos. No garantiza por si solo alineamiento perfecto de estructuras internas H&E-HSI. Debe usarse como referencia clasica frente a futuros metodos deep learning como VoxelMorph o TransMorph.
"""

readme_path = OUTPUT_ROOT / 'README_baseline_clasico.md'
readme_path.write_text(readme, encoding='utf-8')
print('README:', readme_path)
